# Aula 6 (parte 2) — Coleta de Proventos da B3: dividendos, *yield* e armadilhas

## Continuação de `coleta_dados_b3.ipynb`

No notebook anterior coletamos **cotações** (OHLCV) e **macro** (BCB). Ficou faltando a metade da
história: o dinheiro que o ativo **paga** ao investidor.

E não é um detalhe. Naquele notebook descobrimos que o retorno medido por `Close` e por `Adj Close`
difere em **dezenas de pontos percentuais** — e que a diferença é exatamente o que foi distribuído.
Agora vamos buscar esses proventos **um a um**, com data e valor.

### Por que isso merece um notebook inteiro

| Pergunta que só o provento responde | Onde isso é usado |
|---|---|
| Quanto esse ativo me paga por ano, em dinheiro? | estratégias de renda |
| Esse pagamento é **regular** ou foi sorte de um ano? | seleção de carteira |
| O *yield* de 13% é bom ou é um **sinal de alarme**? | detector de armadilha |
| Quanto do retorno veio de **preço** e quanto veio de **provento**? | atribuição de performance |

### O que vamos construir

| Fase | O que fazemos |
|------|---------------|
| **0** | Ambiente e universo de trabalho |
| **1** | Entender o que é um provento (e o que a fonte **não** conta) |
| **2** | Anatomia da API: coletar **um** ticker |
| **3** | Coletar o **universo inteiro**, com robustez |
| **4** | **Auditoria**: o preço confirma os proventos que a fonte informou? |
| **5** | **Dividend Yield** dos últimos 12 meses |
| **6** | O detector de ***dividend trap*** — e por que ele erra |
| **7** | Decompor o retorno total: **preço × provento** |
| **8** | Persistência e coleta incremental |

> ⚠️ **Aviso**: este material é didático, sobre **engenharia de dados**. Nada aqui é recomendação de
> investimento, e as regras tributárias citadas mudam com frequência — confirme a legislação vigente
> antes de usar qualquer número destes para decidir alguma coisa.

---

---

# FASE 0 — AMBIENTE E UNIVERSO

In [ ]:
%pip install -q yfinance pyarrow

In [ ]:
import time
import warnings
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (12, 4)

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("yfinance:", yf.__version__)
print("dados em:", DATA_DIR.resolve())

### O universo: escolhido para haver **contraste**

Diferente do notebook anterior, aqui o universo não é só "papéis líquidos". Ele foi montado para
que os grupos se comportem de forma **visivelmente diferente** — é isso que torna a análise legível.

In [ ]:
# Pagadoras classicas de dividendos: bancos, energia, mineracao, petroleo
ACOES_RENDA = ["PETR4.SA", "VALE3.SA", "BBAS3.SA", "ITUB4.SA", "BBDC4.SA",
               "SANB11.SA", "TAEE11.SA", "EGIE3.SA", "CPFE3.SA", "CMIG4.SA"]

# FIIs: obrigados por lei a distribuir >= 95% do lucro semestral -> pagam TODO MES
FIIS = ["KNRI11.SA", "HGLG11.SA", "XPLG11.SA", "MXRF11.SA", "BTLG11.SA"]

# Grupo de controle: empresas de CRESCIMENTO, que reinvestem em vez de distribuir
CRESCIMENTO = ["WEGE3.SA", "MGLU3.SA"]

UNIVERSO = ACOES_RENDA + FIIS + CRESCIMENTO
GRUPO = ({tk: "acao_renda" for tk in ACOES_RENDA}
         | {tk: "fii" for tk in FIIS}
         | {tk: "crescimento" for tk in CRESCIMENTO})

print(f"{len(UNIVERSO)} ativos:")
for g in ["acao_renda", "fii", "crescimento"]:
    print(f"  {g:12s}: {[t for t in UNIVERSO if GRUPO[t] == g]}")

> 💡 O grupo de **crescimento** (`WEGE3`, `MGLU3`) está aqui de propósito, como **grupo de
> controle**. Sem um contraexemplo, qualquer número parece normal: você só sabe que 9% ao ano é
> muito depois de ver alguém pagando 1,4%.

---

---

# FASE 1 — O QUE É, AFINAL, UM PROVENTO?

## §1.1 — Os quatro tipos que você vai encontrar

| Tipo | O que é | Tributação na pessoa física |
|------|---------|-----------------------------|
| **Dividendo** | parte do **lucro líquido** distribuída em dinheiro | historicamente isento; regra em mudança |
| **JCP** (Juros sobre Capital Próprio) | remuneração do capital, **dedutível** no IR da empresa | **15% retidos na fonte** |
| **Rendimento de FII** | distribuição mensal obrigatória (≥ 95% do lucro semestral) | isento para PF, se o fundo cumprir requisitos legais |
| **Bonificação / desdobramento** | você recebe **mais ações**, não dinheiro | sem efeito de caixa |

### ⚠️ A limitação que mais importa para os seus dados

O Yahoo entrega **um único campo**, chamado `Dividends`. Ele **não separa** dividendo de JCP.

Isso tem consequência direta: como o JCP sofre **15% de retenção na fonte**, o valor que efetivamente
cai na sua conta é menor que o informado. Um cálculo de *yield* líquido feito sobre esses dados vai
**superestimar** a renda de empresas que pagam muito JCP — e bancos brasileiros pagam **muito** JCP.

> 🎯 **Regra**: saiba o que a sua fonte **não** distingue. Toda fonte agrega alguma coisa; o
> problema não é a agregação, é usá-la sem saber que ela existe.

Para separar dividendo de JCP é preciso ir à fonte oficial — os **Comunicados ao Mercado** na CVM /
RI da empresa, ou o sistema **FNET** da B3 no caso de FIIs.

---
## §1.2 — As três datas de um provento

Este é o ponto em que quase todo iniciante se perde, porque **são três datas diferentes**:

| Data | O que significa |
|------|-----------------|
| **Data-com** | último pregão em que comprar o papel **garante** o provento |
| **Data-ex** | primeiro pregão **sem** direito. O preço abre descontado do valor |
| **Data de pagamento** | quando o dinheiro efetivamente cai na conta — pode ser **meses** depois |

### Qual delas o `yfinance` te dá?

A **data-ex**. E isso é exatamente o que você quer, por um motivo sutil: é na data-ex que o **preço
cai**. Casar provento com data-ex mantém preço e provento no mesmo instante — é o que permite a
auditoria da Fase 4 e o cálculo correto de retorno total.

> ⚠️ Se você usasse a **data de pagamento** para calcular retorno, estaria somando um dinheiro que
> entrou em junho a uma queda de preço que aconteceu em março. O retorno total sairia certo no
> acumulado, mas **errado em qualquer janela** menor que o intervalo entre as duas datas.

---

---

# FASE 2 — ANATOMIA DA API: UM TICKER

Antes de coletar 17 ativos, vamos olhar **um** com calma. Essa disciplina — inspecionar o formato
bruto antes de automatizar — evita horas de depuração depois.

In [ ]:
bruto = yf.Ticker("ITUB4.SA").dividends

print("Tipo do objeto :", type(bruto).__name__)
print("Quantidade     :", len(bruto))
print("Nome da serie  :", bruto.name)
print("Tipo do indice :", type(bruto.index).__name__)
print("Timezone       :", bruto.index.tz)
print()
print("Ultimos 5 proventos:")
print(bruto.tail())

### 🔎 Três armadilhas visíveis nessa saída

**1. É uma `Series`, não um `DataFrame`.** O valor está nos dados e a data está no **índice**. Se
você tentar `bruto["Date"]`, recebe um `KeyError`.

**2. O índice tem *timezone* (`America/Sao_Paulo`) e hora (10:00).** Isso parece inofensivo e não é:
um índice *tz-aware* **não compara** com um índice *tz-naive*. Na hora de juntar com a série de
preços (que vem sem fuso), o `merge` simplesmente não encontra nenhuma correspondência — e você
recebe uma tabela vazia, sem erro nenhum. Erro silencioso de novo.

**3. A hora 10:00 também atrapalha.** `2026-09-01 10:00` ≠ `2026-09-01 00:00`. Mesmo depois de
remover o fuso, é preciso **zerar a hora** (`.normalize()`) para o casamento por data funcionar.

A função abaixo resolve os três de uma vez e devolve sempre o **mesmo** formato longo do notebook
anterior: `(Date, Ticker, Dividendo)`.

In [ ]:
COLUNAS = ["Date", "Ticker", "Dividendo"]


def coletar_dividendos_um(ticker: str) -> pd.DataFrame:
    """Devolve os proventos de UM ticker no formato longo (Date, Ticker, Dividendo).

    Nunca levanta excecao: ticker inexistente, delistado ou fora do ar devolve
    DataFrame vazio -- com as colunas certas, para que o concat nao quebre.
    """
    try:
        serie = yf.Ticker(ticker).dividends
    except Exception as e:
        print(f"  [erro] {ticker}: {type(e).__name__} -- {e}")
        return pd.DataFrame(columns=COLUNAS)

    if serie is None or len(serie) == 0:
        return pd.DataFrame(columns=COLUNAS)

    idx = serie.index
    if idx.tz is not None:
        idx = idx.tz_localize(None)     # remove o fuso
    idx = idx.normalize()               # zera a hora (10:00 -> 00:00)

    return pd.DataFrame({"Date": idx, "Ticker": ticker, "Dividendo": serie.values})


exemplo = coletar_dividendos_um("ITUB4.SA")
print(exemplo.shape)
exemplo.tail()

### ⚠️ Por que devolver DataFrame **vazio com as colunas certas**

Repare que o `except` não devolve `None` nem `pd.DataFrame()` pelado: devolve um DataFrame vazio
**com as três colunas**. O motivo é prático — na Fase 3 vamos concatenar dezenas desses. Um
`pd.concat` que recebe um DataFrame sem colunas gera um resultado com colunas desalinhadas, e o erro
só aparece muito depois, na hora de agrupar.

> 💡 Padrão geral: **falha e sucesso devem ter o mesmo formato**. Quem chama a função não precisa
> saber qual dos dois aconteceu para continuar processando.

---

---

# FASE 3 — COLETANDO O UNIVERSO

## §3.1 — Por que aqui é obrigatoriamente um por um

No notebook anterior, `yf.download(lista_de_tickers)` baixava **todos** os preços numa tacada. Com
proventos isso **não existe**: `yf.Ticker(tk).dividends` aceita **um** ticker e faz **uma**
requisição.

Consequência prática: o custo cresce **linearmente** com o universo.

| Universo | Requisições | Tempo aproximado |
|---|---|---|
| 17 ativos (esta aula) | 17 | ~10 segundos |
| 100 ações | 100 | ~1 minuto |
| 1.500 papéis da B3 | 1.500 | **15 a 25 minutos** |

É por isso que a coleta de proventos, num projeto real, é um script **separado** do de cotações e
roda com frequência **menor**. E é por isso que a Fase 8 vai implementar um incremental que evita
rebaixar quem já está em dia.

In [ ]:
t0 = time.time()
partes, vazios = [], []

for i, tk in enumerate(UNIVERSO, start=1):
    df_tk = coletar_dividendos_um(tk)
    if df_tk.empty:
        vazios.append(tk)
    else:
        partes.append(df_tk)
    print(f"[{i:2d}/{len(UNIVERSO)}] {tk:12s} {len(df_tk):4d} proventos", end="")
    print(f"  ({time.time() - t0:5.1f}s)")

dividendos = (pd.concat(partes, ignore_index=True)
                .sort_values(["Ticker", "Date"])
                .reset_index(drop=True))

print(f"\nTotal: {len(dividendos):,} proventos de {dividendos['Ticker'].nunique()} ativos")
print(f"Periodo: {dividendos['Date'].min().date()} a {dividendos['Date'].max().date()}")
print(f"Sem proventos: {vazios if vazios else 'nenhum'}")
print(f"Tempo: {time.time() - t0:.1f}s "
      f"({(time.time() - t0) / len(UNIVERSO):.2f}s por ticker)")

In [ ]:
# Retrato por ativo: quem paga, com que frequencia, desde quando
perfil = (dividendos.groupby("Ticker")
          .agg(n_proventos=("Dividendo", "count"),
               primeiro=("Date", "min"),
               ultimo=("Date", "max"),
               valor_medio=("Dividendo", "mean"),
               valor_max=("Dividendo", "max"))
          .assign(grupo=lambda d: d.index.map(GRUPO))
          .sort_values(["grupo", "n_proventos"], ascending=[True, False]))

perfil["primeiro"] = perfil["primeiro"].dt.date
perfil["ultimo"] = perfil["ultimo"].dt.date
perfil[["grupo", "n_proventos", "primeiro", "ultimo", "valor_medio", "valor_max"]].round(4)

### 🔎 O que a coluna `n_proventos` já revela

- **Bancos** (`ITUB4`, `BBDC4`) aparecem com **centenas** de pagamentos. Não é erro: eles pagam
  dividendo **mensal** — um valor pequeno todo mês — e, por cima, JCP trimestral ou extraordinário
  em valor bem maior. Guarde essa informação: ela vai voltar para nos atrapalhar na Fase 6.
- **FIIs** aparecem com algumas dezenas. Como pagam mensalmente **sem exceção**, o número deveria
  ser bem maior — o que revela um problema de **cobertura da fonte**, que a Fase 4 vai medir.
- **Crescimento** paga pouco e de forma esparsa. É o comportamento esperado de quem reinveste o
  lucro em vez de distribuir.

Compare também `valor_medio` com `valor_max`: quando o máximo é **muitas vezes** a média, existem
pagamentos extraordinários misturados aos ordinários. Isso é normal — mas destrói qualquer métrica
de regularidade calculada ingenuamente.

---

---

# FASE 4 — AUDITORIA: O PREÇO CONFIRMA O PROVENTO?

Esta é a fase mais bonita do notebook, e a ideia é simples.

Sabemos do notebook anterior que o `Adj Close` **já embute** todos os proventos. Então a razão
entre as duas colunas carrega, escondida, a informação de quanto foi pago:

$$f_t = \frac{\text{AdjClose}_t}{\text{Close}_t}
\qquad\Longrightarrow\qquad
D_t = \text{Close}_{t-1}\left(1 - \frac{f_{t-1}}{f_t}\right)$$

Em português: na data-ex, o fator de ajuste **dá um degrau**, e o tamanho do degrau é exatamente o
provento pago.

Ou seja: temos **duas fontes independentes** para a mesma informação — a lista de dividendos e a
série de preços ajustados. Se elas concordarem, a confiança sobe muito. Se discordarem, achamos um
problema **antes** que ele contamine o modelo.

> 🎯 Este é o tipo de checagem que separa quem **usa** dados de quem **confia** neles: validação por
> **redundância**, cruzando dois caminhos que deveriam levar ao mesmo lugar.

In [ ]:
def coletar_precos(tickers, periodo="10y") -> pd.DataFrame:
    """Mesma funcao do notebook anterior, em versao enxuta (Date, Ticker, Close, Adj Close)."""
    bruto = yf.download(tickers, period=periodo, group_by="ticker",
                        auto_adjust=False, progress=False, threads=True)
    partes = []
    for tk in tickers:
        if isinstance(bruto.columns, pd.MultiIndex):
            if tk not in set(bruto.columns.get_level_values(0)):
                continue
            parte = bruto[tk].copy()
        else:
            parte = bruto.copy()
        parte["Ticker"] = tk
        partes.append(parte.reset_index())
    df = pd.concat(partes, ignore_index=True).dropna(subset=["Close"])
    return df[["Date", "Ticker", "Close", "Adj Close"]].sort_values(["Ticker", "Date"])


precos = coletar_precos(UNIVERSO)
print("Precos:", precos.shape)
print(precos["Date"].min().date(), "a", precos["Date"].max().date())
precos.head(3)

In [ ]:
def dividendos_implicitos(px: pd.DataFrame, tolerancia: float = 1e-4) -> pd.DataFrame:
    """Reconstroi os proventos a partir do DEGRAU no fator de ajuste Adj Close / Close.

    A tolerancia existe porque o Adj Close vem arredondado: sem ela, o ruido de
    ponto flutuante faz quase TODO pregao parecer uma data-ex.
    """
    saida = []
    for tk, g in px.groupby("Ticker"):
        g = g.sort_values("Date").reset_index(drop=True)
        fator = g["Adj Close"] / g["Close"]
        razao = fator.shift(1) / fator
        evento = (razao - 1).abs() > tolerancia
        implicito = g["Close"].shift(1) * (1 - razao)
        saida.append(pd.DataFrame({
            "Date": g.loc[evento, "Date"],
            "Ticker": tk,
            "Div_implicito": implicito[evento],
        }))
    return pd.concat(saida, ignore_index=True)


implicitos = dividendos_implicitos(precos)
print(f"Eventos de ajuste detectados no preco: {len(implicitos)}")
print(f"Proventos reportados na mesma janela  : "
      f"{len(dividendos[dividendos['Date'] >= precos['Date'].min()])}")

In [ ]:
# Cruzamento: junta as duas fontes pela chave (Date, Ticker)
janela = dividendos[dividendos["Date"] >= precos["Date"].min()]

conferencia = janela.merge(implicitos, on=["Date", "Ticker"], how="outer", indicator=True)
conferencia["erro_%"] = (conferencia["Div_implicito"] / conferencia["Dividendo"] - 1) * 100

print("Resultado do cruzamento:")
print(conferencia["_merge"].value_counts().rename({
    "both": "confirmado pelas duas fontes",
    "left_only": "reportado, mas SEM ajuste no preco",
    "right_only": "ajuste no preco, mas NAO reportado",
}).to_string())

ok = conferencia[conferencia["_merge"] == "both"]
print(f"\nErro relativo entre as duas fontes (nos {len(ok)} confirmados):")
print(f"  mediano : {ok['erro_%'].abs().median():.4f}%")
print(f"  medio   : {ok['erro_%'].abs().mean():.4f}%")
print(f"  maximo  : {ok['erro_%'].abs().max():.4f}%")

### 📌 Achado: as duas fontes **concordam** — com erro da ordem de 0,01%

A lista de proventos e o ajuste de preço foram gerados de forma consistente pelo provedor. O resíduo
de centésimos de por cento é **arredondamento** do `Adj Close`, que vem com poucas casas decimais.

Mas o interessante está nas exceções. Vamos olhá-las de perto — é ali que a checagem paga o
investimento.

In [ ]:
# Os casos que NAO fecharam
print("--- Proventos reportados SEM ajuste correspondente no preco ---")
orfaos = conferencia[conferencia["_merge"] == "left_only"]
print(orfaos[["Date", "Ticker", "Dividendo"]].to_string(index=False)
      if len(orfaos) else "nenhum")

print("\n--- Os 6 maiores erros relativos entre as duas fontes ---")
piores = ok.reindex(ok["erro_%"].abs().sort_values(ascending=False).index).head(6)
print(piores[["Date", "Ticker", "Dividendo", "Div_implicito", "erro_%"]]
      .round(6).to_string(index=False))

print(f"\nValor mediano dos proventos nos 6 piores casos: R$ {piores['Dividendo'].median():.4f}")
print(f"Valor mediano de TODOS os proventos            : R$ {ok['Dividendo'].median():.4f}")

### 🔎 As exceções não acusam a fonte — acusam o **nosso** detector

**O provento "órfão"** (sem ajuste correspondente) tem valor da ordem de **R$ 0,0005**. Ele não
sumiu do ajuste de preço: o degrau que ele provoca no fator é **menor que a tolerância de 1e-4** que
nós mesmos escolhemos. O detector não o viu porque nós o mandamos ignorar ruído — e, nessa escala,
o provento **é** do tamanho do ruído.

**Os maiores erros relativos** acontecem nos proventos **menores** (compare as duas medianas
impressas acima). Também é aritmética, não defeito: se o `Adj Close` é arredondado na quarta casa,
esse arredondamento representa 2% de um provento de R$ 0,018 e 0,02% de um de R$ 1,80.

> 🎯 **Lição sobre detectores**: antes de acusar o dado, verifique se o achado não é uma propriedade
> do **limiar** que você escolheu. Um alerta de um detector é uma afirmação conjunta sobre o dado
> **e** sobre os parâmetros — e é fácil confundir as duas coisas.

> 💡 A técnica em si vale guardar: sempre que duas colunas do seu dataset são **redundantes por
> construção**, existe uma auditoria de graça esperando para ser escrita.

---
## §4.2 — Concordar não é o mesmo que estar completo

As duas fontes podem estar **erradas juntas**: se um provento antigo nunca entrou na base do
provedor, ele falta na lista **e** no ajuste — e o cruzamento da §4.1 continua fechando lindamente.

Para detectar isso precisamos de outro ângulo. Vamos olhar **quando cada série de proventos
começa**.

In [ ]:
cobertura = pd.DataFrame({
    "grupo": pd.Series(GRUPO),
    "primeiro_preco": precos.groupby("Ticker")["Date"].min(),
    "primeiro_provento": dividendos.groupby("Ticker")["Date"].min(),
    "n_proventos": dividendos.groupby("Ticker")["Dividendo"].count(),
}).dropna(subset=["primeiro_provento"])

print("Data do PRIMEIRO provento reportado, em ordem cronologica:\n")
print(cobertura.sort_values("primeiro_provento")
      .assign(primeiro_provento=lambda d: d["primeiro_provento"].dt.date)
      [["grupo", "primeiro_provento", "n_proventos"]].to_string())

In [ ]:
# Quantos ativos comecam na MESMA data? Coincidencia tem limite.
inicios = cobertura["primeiro_provento"].dt.date.value_counts()
print("Datas de inicio compartilhadas por mais de um ativo:")
print(inicios[inicios > 1].to_string() if (inicios > 1).any() else "nenhuma")

# Para um FII (pagador mensal OBRIGATORIO), quantos pagamentos deveriam existir
# entre o primeiro preco e hoje? E quantos existem de fato?
print("\nFIIs -- pagador mensal por lei, entao 1 provento por mes e o esperado:\n")
for tk in FIIS:
    if tk not in cobertura.index:
        continue
    p0 = cobertura.loc[tk, "primeiro_preco"]          # primeiro preco DESTE ticker
    d0 = cobertura.loc[tk, "primeiro_provento"]
    meses_cobertos = (precos["Date"].max() - p0).days / 30.44
    lacuna = (d0 - p0).days / 30.44
    n = int(cobertura.loc[tk, "n_proventos"])
    print(f"  {tk}: preco desde {p0.date()} ({meses_cobertos:5.1f} meses) | "
          f"proventos: {n:3d} | lacuna inicial: {lacuna:5.1f} meses | "
          f"faltam ~{max(0, round(meses_cobertos) - n):3d}")

### ⚠️ Achado: o histórico de proventos é **truncado** — e a prova é a coincidência

Olhe as datas de início. **Três FIIs diferentes** começam a pagar exatamente no mesmo dia
(**01/11/2017**), e um quarto duas semanas antes. Fundos independentes, com gestoras diferentes,
patrimônios diferentes e datas de constituição diferentes.

Isso não é comportamento de mercado — é **assinatura de corte na base de dados**. A explicação
honesta é que o provedor só tem histórico de proventos a partir dali.

O mesmo padrão aparece nas ações, agrupadas no início de **2008**. E lembre que os FIIs da lista
pagam mensalmente **desde a sua constituição**, vários deles anteriores a 2017: os proventos
anteriores simplesmente **não estão lá** — nem na lista, nem no ajuste do `Adj Close`, como a §4.1
já tinha mostrado ao encontrar o mesmo número de eventos nas duas fontes.

### O que isso significa na prática

1. O **retorno total de longo prazo** calculado com `Adj Close` está **subestimado** para esses
   ativos. Quanto mais antiga a janela, pior.
2. Um **DY histórico** calculado em janelas antigas vai dar perto de zero — e a conclusão "esse FII
   não pagava nada em 2016" seria **falsa**.
3. Comparar o desempenho de um FII com o de uma ação em janela longa fica **enviesado contra o FII**,
   porque é dele que falta mais provento.

> 🎯 **A regra que sobra desta fase**: restrinja a análise à janela em que a cobertura de proventos
> é confiável — aqui, aproximadamente de **2018 em diante**. É por isso que, da Fase 5 em diante,
> todo cálculo usa uma janela recente. Não é preguiça: é o limite do dado.

> 💡 Precisa do histórico completo? A fonte oficial de FIIs é o sistema **FNET** da B3, e a de ações
> são os comunicados de RI / CVM. Ambos exigem bem mais trabalho — e é assim que costuma ser: dado
> confiável é caro, e o grátis vem com pegadinha.

---

---

# FASE 5 — DIVIDEND YIELD DOS ÚLTIMOS 12 MESES

O **DY** (*dividend yield*) responde: *quanto esse ativo me pagou, em proporção ao que ele custa?*

$$DY_{12m} = \frac{\text{soma dos proventos dos últimos 12 meses}}{\text{preço atual}} \times 100$$

Parece trivial. Tem três decisões escondidas — e é nelas que o número ganha ou perde sentido.

In [ ]:
data_ref = precos["Date"].max()
corte_12m = data_ref - pd.DateOffset(months=12)

# DECISAO 1: preco ATUAL (nao o medio do periodo) -- e o preco que voce pagaria hoje
preco_atual = precos.sort_values("Date").groupby("Ticker")["Close"].last()

# DECISAO 2: proventos dos ultimos 12 meses pela DATA-EX
div_12m = dividendos[(dividendos["Date"] > corte_12m) & (dividendos["Date"] <= data_ref)]

dy = (div_12m.groupby("Ticker")["Dividendo"]
      .agg(soma_12m="sum", n_pagamentos="count", media="mean", desvio="std"))
dy["preco"] = preco_atual
dy["DY_12m_%"] = dy["soma_12m"] / dy["preco"] * 100

# DECISAO 3: CV = desvio / media -- uma medida de REGULARIDADE dos pagamentos
dy["CV"] = (dy["desvio"] / dy["media"]).where(dy["n_pagamentos"] >= 4)
dy["grupo"] = dy.index.map(GRUPO)

print(f"Janela: {corte_12m.date()} a {data_ref.date()}\n")
dy.sort_values("DY_12m_%", ascending=False)[
    ["grupo", "preco", "n_pagamentos", "soma_12m", "DY_12m_%", "CV"]].round(3)

### 🔎 As três decisões, e por que cada uma importa

**1. Preço atual, não médio.** O DY existe para responder *"se eu comprar hoje, quanto recebo?"*.
Usar o preço médio do período mistura um pagamento que já aconteceu com um preço que não existe
mais.

**2. Data-ex, não data de pagamento.** Como vimos na Fase 1, é a data-ex que casa provento e preço
no mesmo instante.

**3. *Trailing*, não *forward*.** Estamos olhando o que **já** foi pago (*trailing*). O mercado
costuma usar o DY **projetado** (*forward*), que depende de estimativa de lucro futuro — e
estimativa não é dado coletado, é opinião. Fique com o *trailing*: ele é verificável.

### ⚠️ A aritmética que engana

$DY = D/P$ sobe de duas maneiras opostas:

- **$D$ subiu** — a empresa passou a pagar mais. Ótimo.
- **$P$ caiu** — o mercado derrubou o preço. Péssimo.

**A fórmula não distingue as duas.** Um DY que dispara costuma ser o **denominador** desabando, e
não o numerador crescendo. Vamos separar os dois efeitos.

In [ ]:
# Qual parte do DY atual vem de preco e qual vem de dividendo?
# Truque: recalcular o MESMO DY usando o preco de 12 meses atras.
precos_ord = precos.sort_values("Date")
preco_ha_12m = (precos_ord[precos_ord["Date"] <= corte_12m]
                .groupby("Ticker")["Close"].last())

comp = pd.DataFrame({
    "DY_com_preco_hoje_%": dy["soma_12m"] / preco_atual * 100,
    "DY_com_preco_12m_%": dy["soma_12m"] / preco_ha_12m * 100,
    "var_preco_%": (preco_atual / preco_ha_12m - 1) * 100,
}).dropna()
comp["efeito_preco_pp"] = comp["DY_com_preco_hoje_%"] - comp["DY_com_preco_12m_%"]
comp["grupo"] = comp.index.map(GRUPO)

print("Mesmos proventos, precos diferentes -- o DY muda so por causa do denominador:\n")
print(comp.sort_values("efeito_preco_pp", ascending=False).round(2).to_string())

### 📌 Leitura: o denominador manda no indicador

A coluna `efeito_preco_pp` isola quanto do DY atual vem **puramente** da variação do preço, com os
proventos mantidos fixos. Onde ela é **positiva e grande**, o *yield* subiu porque a **cotação
caiu** — o ativo não ficou mais generoso, ficou mais barato. E quando o preço cai, a pergunta certa
não é "que *yield* bonito", e sim **"por que caiu?"**.

> 🎯 Um indicador que é uma **razão** sempre pode se mover pelo numerador ou pelo denominador.
> Antes de interpretar qualquer razão — DY, P/L, margem, taxa de conversão — separe os dois. Essa
> é uma técnica geral de análise, não uma particularidade de finanças.

---

---

# FASE 6 — O DETECTOR DE ARMADILHA (E POR QUE ELE FALHA)

Existe um padrão folclórico no mercado: *"yield alto demais é sinal de armadilha"* — a **dividend
trap**, em que o alto *yield* é fruto de um preço derretendo ou de uma distribuição insustentável.

Vamos **implementar** esse detector, no formato usado em pipelines reais: um **score** que soma
penalidade por *yield* excessivo e penalidade por irregularidade.

In [ ]:
def score_risco(dy_pct, cv, dy_neutro=10.0, cv_neutro=0.5):
    """Score heuristico de risco de distribuicao.

    Acima de `dy_neutro` cada 10 pontos percentuais de DY somam 1.0.
    Acima de `cv_neutro` cada 1.0 de CV soma 1.0. Total > 1.0 = alerta.
    """
    if pd.isna(dy_pct):
        return np.nan
    s_dy = max(0.0, (dy_pct - dy_neutro) / 10.0)
    s_cv = 0.0 if pd.isna(cv) else max(0.0, (cv - cv_neutro) / 1.0)
    return round(s_dy + s_cv, 2)


dy["score_risco"] = [score_risco(r["DY_12m_%"], r["CV"]) for _, r in dy.iterrows()]
dy["alerta"] = dy["score_risco"] > 1.0

print("Ranking do detector (maior score = supostamente mais arriscado):\n")
print(dy.sort_values("score_risco", ascending=False)[
    ["grupo", "DY_12m_%", "CV", "n_pagamentos", "score_risco", "alerta"]]
    .round(3).to_string())

### 🚨 O detector elegeu como "mais arriscado"... o pagador mais regular da lista

Se a sua saída for parecida com a minha, no topo do ranking estão **`ITUB4`** e **`BBDC4`** — dois
dos maiores bancos do país, que pagam proventos **todo mês**, sem interrupção, há mais de uma
década. E lá embaixo, tranquilo, está um FII com DY de **13%**.

O detector não está com bug. Ele está **medindo a coisa errada**. Vamos ver com os próprios olhos
por que o CV desses bancos explodiu.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, tk in zip(axes, ["ITUB4.SA", "MXRF11.SA"]):
    sub = dividendos[(dividendos["Ticker"] == tk) &
                     (dividendos["Date"] >= data_ref - pd.DateOffset(years=3))]
    ax.bar(sub["Date"], sub["Dividendo"], width=8)
    cv_tk = dy.loc[tk, "CV"]
    ax.set(title=f"{tk} - proventos dos ultimos 3 anos (CV = {cv_tk:.2f})",
           xlabel="Data-ex", ylabel="R$ por acao/cota")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# O mesmo fenomeno, em numeros: a distribuicao dos VALORES pagos
for tk in ["ITUB4.SA", "MXRF11.SA"]:
    sub = dividendos[(dividendos["Ticker"] == tk) & (dividendos["Date"] > corte_12m)]
    v = sub["Dividendo"]
    print(f"{tk}: {len(v)} pagamentos em 12 meses")
    print(f"   menor = R$ {v.min():.4f} | mediana = R$ {v.median():.4f} | maior = R$ {v.max():.4f}")
    print(f"   razao maior/menor = {v.max() / v.min():.0f}x\n")

### 📌 Diagnóstico: o CV mede **irregularidade**, não **risco**

O Itaú paga **duas coisas diferentes** no mesmo campo:

- um dividendo **mensal** minúsculo (centavos por ação), todo mês, como um relógio;
- **JCP** trimestral ou extraordinário, dezenas de vezes maior.

O desvio-padrão sobre essa mistura é gigante — e o CV, que é desvio dividido pela média, vai junto.
Mas isso descreve o **formato do calendário de pagamentos**, não a saúde da empresa. O FII, que paga
sempre o mesmo valor, tem CV perto de zero — o que também não diz nada sobre a qualidade dos
imóveis dele.

### As três falhas do detector, nomeadas

| Falha | Por quê |
|-------|---------|
| **CV confunde mistura com instabilidade** | frequências e tipos diferentes no mesmo campo inflam o desvio |
| **CV não é comparável entre regimes** | mensal, trimestral e anual produzem CVs em escalas diferentes |
| **DY alto não implica insustentável** | pode ser um ativo barato, ou um FII com contratos longos |

### O que seria preciso para responder de verdade

Saber se uma distribuição é sustentável exige dado **fundamentalista**, que **não está** no nosso
conjunto: *payout ratio*, geração de caixa, FFO, vacância, endividamento, receita contratada. Tudo
isso vem de balanços e relatórios gerenciais — não de uma série de preços.

> 🎯 **O resultado desta fase é negativo, e é o mais valioso do notebook**: o dado que temos **não
> responde** à pergunta que quisemos fazer. Reconhecer isso é análise. Publicar o ranking do
> detector como se ele medisse risco seria produzir um número com aparência de conhecimento —
> e alguém tomaria decisão com ele.

### O que dá para salvar

A primeira coisa a fazer é **segmentar por regime de pagamento** antes de comparar qualquer coisa.
Comparar o CV de um pagador mensal com o de um pagador anual é comparar coisas diferentes.

In [ ]:
# Regime de pagamento: em quantos dos ultimos 12 meses houve provento?
meses = pd.date_range(corte_12m.normalize(), data_ref.normalize(), freq="ME")

regularidade = {}
for tk, sub in div_12m.groupby("Ticker"):
    m = sub.set_index("Date")["Dividendo"].resample("ME").sum().reindex(meses, fill_value=0)
    regularidade[tk] = (m > 0).mean()

dy["meses_com_pagto"] = pd.Series(regularidade)
dy["regime"] = np.where(dy["meses_com_pagto"] >= 0.75, "mensal", "esporadico")

print("Comparar CV so faz sentido DENTRO de cada regime:\n")
for reg in ["mensal", "esporadico"]:
    bloco = dy[dy["regime"] == reg].sort_values("CV")
    print(f"--- regime {reg.upper()} ---")
    print(bloco[["grupo", "DY_12m_%", "n_pagamentos", "meses_com_pagto", "CV"]]
          .round(3).to_string())
    print()

### 📌 Agora sim a comparação tem sentido

Dentro do regime **mensal**, o CV volta a significar alguma coisa: os FIIs, que pagam valores quase
idênticos todo mês, ficam com CV perto de zero; os bancos, que misturam mensal miúdo com JCP graúdo,
ficam altos — e agora sabemos **por quê**, em vez de sermos enganados pelo número.

No regime **esporádico**, o CV entre 3 e 6 pagamentos anuais é estatisticamente frágil: com tão
poucas observações, um pagamento extraordinário domina a conta inteira.

> 💡 **Segmentar antes de comparar** é uma das ideias mais transferíveis desta aula. Vale para
> churn por plano, conversão por canal, latência por região — sempre que a população mistura
> regimes diferentes, a média (e o desvio) de tudo junto não descreve ninguém.

---

---

# FASE 7 — DECOMPONDO O RETORNO TOTAL

No notebook anterior vimos que `Close` e `Adj Close` contam histórias diferentes. Agora podemos
**quantificar a diferença** e dar nome a ela.

Se $r_{total}$ é o retorno com proventos reinvestidos e $r_{preço}$ é a valorização pura:

$$(1 + r_{total}) = (1 + r_{preço}) \times (1 + r_{provento})
\qquad\Longrightarrow\qquad
r_{provento} = \frac{1 + r_{total}}{1 + r_{preço}} - 1$$

Os retornos **compõem** (multiplicam), não somam — pelo mesmo motivo visto na aula anterior com as
séries de inflação.

> ⚠️ Vamos usar uma janela de **5 anos**, não de 10. Não é escolha estética: a Fase 4 mostrou que a
> cobertura de proventos antes de ~2018 é incompleta, e uma janela longa **subestimaria** a parcela
> de provento justamente nos ativos que mais pagam.

In [ ]:
inicio = data_ref - pd.DateOffset(years=5)
janela = precos[precos["Date"] >= inicio]

linhas = []
for tk, g in janela.groupby("Ticker"):
    g = g.sort_values("Date")
    if len(g) < 100:
        continue
    r_preco = g["Close"].iloc[-1] / g["Close"].iloc[0] - 1
    r_total = g["Adj Close"].iloc[-1] / g["Adj Close"].iloc[0] - 1
    r_prov = (1 + r_total) / (1 + r_preco) - 1
    linhas.append({
        "Ticker": tk,
        "grupo": GRUPO[tk],
        "ret_preco_%": r_preco * 100,
        "ret_provento_%": r_prov * 100,
        "ret_total_%": r_total * 100,
        "parcela_provento_%": r_prov / r_total * 100 if r_total > 0 else np.nan,
    })

decomp = pd.DataFrame(linhas).set_index("Ticker").sort_values("ret_total_%", ascending=False)
print(f"Janela: {janela['Date'].min().date()} a {data_ref.date()} (5 anos)\n")
decomp.round(1)

In [ ]:
ordem = decomp.sort_values("ret_total_%").index
y = np.arange(len(ordem))
altura = 0.38

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(y - altura / 2, decomp.loc[ordem, "ret_preco_%"], height=altura,
        label="valorizacao do preco")
ax.barh(y + altura / 2, decomp.loc[ordem, "ret_provento_%"], height=altura,
        label="proventos")
ax.plot(decomp.loc[ordem, "ret_total_%"], y, "D", color="black", ms=5,
        label="retorno total (composto)")

ax.set_yticks(y, ordem)
ax.axvline(0, color="black", lw=0.8)
ax.set(title="De onde veio o retorno de 5 anos", xlabel="Retorno acumulado (%)")
ax.legend()
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

### 📌 O que a decomposição mostra

- **FIIs**: a barra de preço é curta — às vezes **negativa** — e quase todo o retorno vem da barra
  de proventos. Faz sentido: um FII distribui obrigatoriamente ≥ 95% do lucro semestral, então
  sobra pouco lucro retido para valorizar a cota. Quem olha só a cotação de um FII conclui que ele
  "não anda". Ele não anda mesmo — ele **paga**.
- **Ações de renda**: retorno repartido entre as duas fontes.
- **Crescimento**: a barra de proventos é quase invisível. O lucro voltou para a empresa, e o
  retorno (quando existe) aparece no preço.

> 🎯 Aqui fecha o argumento do notebook anterior: **usar `Close` para medir retorno de FII é
> simplesmente medir a coisa errada** — descarta a parcela que é praticamente o retorno inteiro.

### 🔎 Por que as barras estão **lado a lado** e não empilhadas

Porque a decomposição é **multiplicativa**: $(1+r_{total}) = (1+r_{pre	ext{ç}o})(1+r_{provento})$.
As duas parcelas **não somam** o total — empilhá-las desenharia uma soma que não existe, e o
leitor mediria a barra com a régua achando que está lendo o total. O losango marca o retorno total
de verdade.

> 💡 Sempre que a sua conta não for uma soma, **não use gráfico de barras empilhadas**. A geometria
> de um gráfico é uma afirmação matemática, e o leitor vai acreditar nela antes de ler a legenda.

### ⚠️ Duas leituras da coluna `parcela_provento_%`

- **Acima de 100%**: a valorização do preço foi **negativa**. O provento carregou o retorno sozinho
  e ainda cobriu a perda de cotação — é o caso de vários FIIs aqui.
- **Nula**: o retorno total foi **negativo ou zero**. Aí a razão perderia sentido (uma fração de um
  número negativo produz um percentual com sinal enganoso), então a coluna é deixada vazia de
  propósito — não é dado faltando.

---

---

# FASE 8 — PERSISTÊNCIA E COLETA INCREMENTAL

## §8.1 — Por que o incremental aqui é diferente

No notebook de cotações, o incremental era simples: guardar a última data e pedir só o que veio
depois. Para proventos isso **não funciona**, porque a API não aceita data inicial — `.dividends`
devolve **o histórico inteiro**, sempre.

Então onde está a economia? Não em baixar menos por ticker, e sim em **não visitar tickers
desnecessariamente**. Para isso precisamos registrar *quando* cada ticker foi consultado pela última
vez — e isso **não cabe** na tabela de proventos, que só tem os pagamentos que existem.

A solução é um **arquivo de controle** separado: uma linha por ticker, com a data da última
verificação.

| Arquivo | Conteúdo | Papel |
|---------|----------|-------|
| `b3_dividendos.parquet` | `Date, Ticker, Dividendo` | os dados |
| `b3_dividendos_controle.parquet` | `Ticker, ultima_verificacao` | o **estado da coleta** |

> 💡 Separar **dado** de **estado da coleta** é um padrão que reaparece em todo pipeline sério:
> *checkpoints*, *watermarks*, tabelas de controle de ETL. O dado responde "o que é verdade"; o
> controle responde "até onde eu já fui".

In [ ]:
CAMINHO_DIV = DATA_DIR / "b3_dividendos.parquet"
CAMINHO_CTRL = DATA_DIR / "b3_dividendos_controle.parquet"


def atualizar_dividendos(tickers, dias_validade: int = 7,
                         checkpoint_a_cada: int = 25) -> pd.DataFrame:
    """Coleta proventos visitando apenas os tickers cuja verificacao expirou.

    Args:
        tickers:          universo desejado.
        dias_validade:    por quantos dias uma verificacao continua valendo.
        checkpoint_a_cada: salva parcial a cada N tickers (protege contra queda no meio).
    """
    dados = (pd.read_parquet(CAMINHO_DIV) if CAMINHO_DIV.exists()
             else pd.DataFrame(columns=COLUNAS))
    ctrl = (pd.read_parquet(CAMINHO_CTRL) if CAMINHO_CTRL.exists()
            else pd.DataFrame(columns=["Ticker", "ultima_verificacao"]))

    visto = dict(zip(ctrl["Ticker"], pd.to_datetime(ctrl["ultima_verificacao"])))
    limite = pd.Timestamp.today().normalize() - pd.Timedelta(days=dias_validade)

    pendentes = [tk for tk in tickers
                 if tk not in visto or visto[tk] < limite]
    print(f"{len(tickers)} no universo | {len(tickers) - len(pendentes)} ainda validos "
          f"(verificados ha menos de {dias_validade}d) | {len(pendentes)} a consultar")

    if not pendentes:
        print("Nada a fazer -- nenhuma requisicao foi feita.")
        return dados

    hoje = pd.Timestamp.today().normalize()
    for i, tk in enumerate(pendentes, start=1):
        novo = coletar_dividendos_um(tk)
        if not novo.empty:
            dados = pd.concat([dados, novo], ignore_index=True)
            # keep='last': o recem-baixado sobrescreve a versao antiga da mesma data
            dados = dados.drop_duplicates(subset=["Date", "Ticker"], keep="last")
        visto[tk] = hoje

        if i % checkpoint_a_cada == 0 or i == len(pendentes):
            dados.sort_values(["Ticker", "Date"]).to_parquet(CAMINHO_DIV, index=False)
            print(f"  checkpoint em {i}/{len(pendentes)} ({len(dados):,} proventos salvos)")

    dados = dados.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    dados.to_parquet(CAMINHO_DIV, index=False)
    (pd.DataFrame({"Ticker": list(visto), "ultima_verificacao": list(visto.values())})
       .to_parquet(CAMINHO_CTRL, index=False))
    return dados


print("--- Execucao 1 (base vazia) ---")
base_div = atualizar_dividendos(UNIVERSO)
print(f"\nBase: {len(base_div):,} proventos")

In [ ]:
# Execucao 2, imediatamente depois: TODOS estao dentro da validade.
# Zero requisicoes -- e a base continua identica. Idempotencia na pratica.
print("--- Execucao 2 (logo em seguida) ---")
antes = len(base_div)
base_div = atualizar_dividendos(UNIVERSO)
print(f"\nBase: {antes:,} -> {len(base_div):,} proventos (delta {len(base_div) - antes:+d})")

In [ ]:
# Execucao 3: um ticker novo entra no universo. So ele e consultado.
print("--- Execucao 3 (universo ampliado) ---")
base_div = atualizar_dividendos(UNIVERSO + ["CSMG3.SA"])
print(f"\nBase: {len(base_div):,} proventos de {base_div['Ticker'].nunique()} ativos")

### 📌 O ganho, em números

Com 17 tickers a economia parece modesta. Projete para o universo real da B3:

| Cenário | Requisições | Tempo |
|---------|-------------|-------|
| Sem controle, todo dia | 1.500 | ~20 min/dia |
| Com validade de 7 dias | ~210/dia (1.500 ÷ 7) | ~3 min/dia |

E há um ganho que não aparece na tabela: **menos requisições, menos chance de bloqueio**. Um serviço
gratuito que você martela 1.500 vezes por dia acaba te respondendo com erro — e aí você não tem
dado nenhum.

> ⚠️ `dias_validade` é uma **escolha de negócio**, não técnica. FIIs pagam todo mês: uma validade de
> 7 dias nunca perde um pagamento por muito tempo. Se o seu caso exige saber do provento **no dia**,
> a validade tem que ser 0 e o custo volta. Não existe resposta certa fora do contexto.

In [ ]:
# Resumo legivel para quem nao vai abrir o parquet (ex.: alguem revisando por cima)
resumo = (base_div.groupby("Ticker")
          .agg(n_proventos=("Dividendo", "count"),
               primeiro=("Date", "min"),
               ultimo=("Date", "max"),
               soma_total=("Dividendo", "sum"))
          .sort_values("n_proventos", ascending=False))
resumo["primeiro"] = resumo["primeiro"].dt.date
resumo["ultimo"] = resumo["ultimo"].dt.date

caminho_resumo = DATA_DIR / "b3_dividendos_resumo.csv"
resumo.to_csv(caminho_resumo, sep=";", encoding="utf-8-sig")   # ; e BOM: Excel BR abre direito

print("Arquivos gerados:\n")
for f in sorted(DATA_DIR.glob("b3_dividendos*")):
    print(f"  {f.name:36s} {f.stat().st_size / 1024:8.1f} KB")
print()
resumo.head(8)

### 💡 Por que `sep=";"` e `encoding="utf-8-sig"` no CSV

Detalhe pequeno que evita uma conversa chata depois. O Excel em português brasileiro:

- usa **ponto e vírgula** como separador de colunas (a vírgula é o separador **decimal**);
- só reconhece UTF-8 se o arquivo tiver **BOM** — que é o que o `utf-8-sig` acrescenta. Sem ele,
  "Alocação" vira "AlocaÃ§Ã£o".

Os dados de verdade ficam no Parquet. Esse CSV existe só para **ser aberto por uma pessoa** — e por
isso ele tem que abrir bem na ferramenta que a pessoa usa.

---

---

# 🎓 FECHAMENTO

### O que você construiu

| Componente | Função |
|------------|--------|
| `coletar_dividendos_um()` | proventos de um ticker, com fuso e hora normalizados |
| `dividendos_implicitos()` | **reconstrói** os proventos a partir do ajuste de preço |
| cruzamento das duas fontes | auditoria por **redundância** |
| cálculo de DY 12m + decomposição | *yield* e a separação preço × dividendo |
| `score_risco()` | o detector de armadilha — e a **demolição** dele |
| decomposição do retorno | quanto veio de preço, quanto veio de provento |
| `atualizar_dividendos()` | incremental com **arquivo de controle** |

### As seis ideias que sobrevivem a esta aula

1. **Saiba o que a sua fonte não distingue.** Dividendo e JCP chegam no mesmo campo, com tributação
   diferente.
2. **Fuso horário e hora no índice quebram `merge` em silêncio.** Normalize antes de juntar.
3. **Redundância é auditoria de graça.** Se duas colunas contêm a mesma informação por construção,
   cruze as duas.
4. **Concordância não é completude.** As duas fontes podiam estar — e estavam — truncadas juntas.
5. **Toda razão se move por dois lados.** DY subindo pode ser dividendo crescendo ou preço caindo.
6. **Segmente antes de comparar.** CV de pagador mensal e de pagador anual não vivem na mesma escala.

### E a lição mais desconfortável

O detector de *dividend trap* da Fase 6 **não funciona** — e descobrir isso foi o melhor resultado
do notebook. Havia duas saídas possíveis quando o ranking apontou o Itaú como o ativo mais
arriscado: ajustar os parâmetros até a saída ficar "razoável", ou investigar por que o indicador se
comportou assim. A primeira produz um número apresentável e falso.

> 🎯 Um pipeline de dados bem construído não é o que sempre dá resposta. É o que **deixa visível
> quando não dá** — e aqui a auditoria da Fase 4 e a segmentação da Fase 6 fizeram exatamente isso.

### Juntando os dois notebooks

Você agora tem, em `data/raw/`:

| Arquivo | Conteúdo |
|---------|----------|
| `b3_cotacoes.parquet` | OHLCV diário por ativo |
| `b3_macro.parquet` | IPCA, IGP-M, INCC, Selic |
| `b3_dividendos.parquet` | proventos por data-ex |
| `b3_dividendos_controle.parquet` | estado da coleta |

É a base completa para a próxima etapa: **engenharia de atributos** e **modelagem** — com a
diferença de que agora você sabe exatamente onde cada número foi buscado, e onde ele mente.

---

---

# 📝 EXERCÍCIOS

### Nível 1 — aquecimento

**1.1** Acrescente três pagadoras de dividendos ao `UNIVERSO` (ex.: `CSMG3.SA`, `CXSE3.SA`,
`TRPL4.SA`), rode `atualizar_dividendos()` e confirme que **apenas as três novas** foram
consultadas.

**1.2** Calcule o DY de **cada um dos últimos 5 anos** (não só dos últimos 12 meses) para os FIIs do
universo. O DY deles é estável ao longo do tempo?

**1.3** Monte um calendário: para `KNRI11.SA`, mostre em que **dia do mês** costuma cair a data-ex.
O padrão é estável?

### Nível 2 — análise

**2.1** Refaça a decomposição da Fase 7 usando janelas de **1, 3, 5 e 10 anos**. A parcela vinda de
proventos cresce com a janela — e o que a Fase 4 diz sobre confiar no número de 10 anos?

**2.2** Implemente `dy_liquido()` supondo que **todo** provento de ação seja JCP (15% retidos) e
que rendimento de FII seja isento. Quanto muda o ranking da Fase 5? Quem sobe, quem desce?

**2.3** Para cada ativo, calcule o **retorno de preço nos 5 pregões seguintes à data-ex**. Em média,
o papel recupera a queda do provento? Repita separando FIIs de ações — o comportamento é diferente?

### Nível 3 — engenharia

**3.1** Estenda `dividendos_implicitos()` para detectar **desdobramentos** usando
`yf.Ticker(tk).splits`. Dica: verifique primeiro se o Yahoo já aplica o split na coluna `Close` —
a resposta muda todo o exercício, e descobrir isso **é** o exercício.

**3.2** Escreva `validar_dividendos(div, precos)` que rode a auditoria da Fase 4 e **levante
`ValueError`** se mais de 5% dos proventos não tiverem ajuste de preço correspondente. Transforme
a checagem manual em teste automático.

**3.3** Junte os dois notebooks num módulo `coleta_b3.py` com `main()` que atualize cotações, macro
e proventos em sequência, gravando um log com o tempo de cada etapa.

### Nível 4 — desafio

**4.1** Rode a coleta de proventos para **todos os FIIs** do Fundamentus (use a função da aula
anterior). Implemente lotes com pausa e *retry* com espera exponencial. Quantos FIIs não têm nenhum
provento registrado no Yahoo — e o que isso diz sobre a fonte?

**4.2** Pesquise o **FNET** da B3 (informes de rendimento de FIIs, em XML/JSON) e escreva um
coletor. Compare, para `KNRI11`, o histórico oficial com o do Yahoo: quantos meses faltam no Yahoo?
Confirme ou refute a conclusão da Fase 4.

**4.3** Construa um **universo dinâmico**, no espírito de um pipeline de produção: dada uma data
$D_0$, selecione as 30 ações com DY dos 12 meses **anteriores a $D_0$** ≥ 5%, liquidez média diária
≥ R$ 500 mil e cobertura de preços ≥ 80%. O ponto crítico: nenhuma informação posterior a $D_0$
pode entrar na seleção. Explique, em duas linhas, onde o *look-ahead bias* entraria se você
descuidasse.

---